# Lab 19-02 · Inverter Delay, Ring Oscillators and Supply Scaling

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import delay as dl

## Part 1 · Step delays

*Predict* t_pHL and t_pLH for Example 19-02.2 (20 fF, kn = 200 µA/V², kp = 100 µA/V², VDD = 1.2 V, thresholds 0.35 V), then compare with `ngspice -b delay.cir`.

In [2]:
C, VDD, VT = 20e-15, 1.2, 0.35
thl = dl.tp_step_exact(C, 200e-6, VDD, VT)
tlh = dl.tp_step_exact(C, 100e-6, VDD, VT)
print(f"t_pHL closed form {thl * 1e12:.2f} ps, ODE {dl.tp_step_ode(C, 200e-6, VDD, VT) * 1e12:.2f} ps")
print(f"t_pLH {tlh * 1e12:.2f} ps, t_p {(thl + tlh) / 2 * 1e12:.2f} ps")
req = dl.r_eq(200e-6, VDD, VT)
print(f"R_eq {req / 1e3:.3f} kOhm (approximation {dl.r_eq_approx(200e-6, VDD, VT) / 1e3:.3f}); RC delay with 12.57 kOhm {dl.tp_rc(12.57e3, C) * 1e12:.1f} ps")

t_pHL closed form 168.20 ps, ODE 168.20 ps
t_pLH 336.39 ps, t_p 252.29 ps
R_eq 12.573 kOhm (approximation 12.457); RC delay with 12.57 kOhm 174.3 ps


## Part 2 · Scaling with load and strength

*Predict* t_pHL when the load doubles, and when kn doubles.

In [3]:
for c, k in ((20e-15, 200e-6), (40e-15, 200e-6), (20e-15, 400e-6)):
    print(f"C = {c * 1e15:.0f} fF, kn = {k * 1e6:.0f} uA/V^2: t_pHL = {dl.tp_step_exact(c, k, VDD, VT) * 1e12:.1f} ps")

C = 20 fF, kn = 200 uA/V^2: t_pHL = 168.2 ps
C = 40 fF, kn = 200 uA/V^2: t_pHL = 336.4 ps
C = 20 fF, kn = 400 uA/V^2: t_pHL = 84.1 ps


## Part 3 · The RC model's error

*Predict* whether the ratio of the RC delay to the exact delay depends on C or k.

In [4]:
x = np.linspace(0.2, 0.45, 26)
ratio = [dl.tp_rc(dl.r_eq(1.0, 1.0, xi), 1.0) / dl.tp_step_exact(1.0, 1.0, 1.0, xi) for xi in x]
plt.plot(x, ratio); plt.xlabel("VTN / VDD"); plt.ylabel("RC delay / exact delay"); plt.show()
print(f"ratio at 0.2: {ratio[0]:.4f}, at 0.45: {ratio[-1]:.4f}")
print("same ratio at another C, k and VDD:", round(dl.tp_rc(dl.r_eq(7e-4, 2.0, 0.8), 3e-15) / dl.tp_step_exact(3e-15, 7e-4, 2.0, 0.8), 4), "(VTN/VDD = 0.4)")

ratio at 0.2: 1.0324, at 0.45: 1.0396
same ratio at another C, k and VDD: 1.0391 (VTN/VDD = 0.4)


## Part 4 · Fan-out and sizing

*Predict* where each curve starts and how its slope depends on the size S (Example 19-02.4: R_eq = 12.57 kΩ, C_int = 2 fF).

In [5]:
tp0 = dl.tp_rc(12.57e3, 2e-15)
cext = np.linspace(0, 50e-15, 101)
for S in (1, 2, 4):
    plt.plot(cext * 1e15, [dl.tp_with_fanout(tp0, c, 2e-15, S) * 1e12 for c in cext], label=f"S = {S}")
    print(f"S = {S}: {dl.tp_with_fanout(tp0, 50e-15, 2e-15, S) * 1e12:.1f} ps at 50 fF")
plt.xlabel("external load (fF)"); plt.ylabel("t_p (ps)"); plt.legend(); plt.show()
print(f"intrinsic delay {tp0 * 1e12:.2f} ps")

S = 1: 453.1 ps at 50 fF
S = 2: 235.2 ps at 50 fF
S = 4: 126.3 ps at 50 fF
intrinsic delay 17.43 ps


## Part 5 · Ring oscillators

*Predict* the period with seven stages instead of five, and whether the stage delay changes.

In [6]:
step = dl.tp_step_exact(10e-15, 200e-6, VDD, VT)
for n in (5, 7):
    per = dl.simulate_ring(n, 200e-6, 200e-6, VDD, VT, VT, 10e-15)
    print(f"{n} stages: period {per * 1e9:.4f} ns, stage delay {per / (2 * n) * 1e12:.2f} ps, "
          f"{per / (2 * n) / step:.3f} times the step delay {step * 1e12:.2f} ps")

5 stages: period 1.6050 ns, stage delay 160.50 ps, 1.909 times the step delay 84.10 ps


7 stages: period 2.2471 ns, stage delay 160.51 ps, 1.909 times the step delay 84.10 ps


## Part 6 · Supply scaling

*Predict* where the EDP minimum lies for alpha = 1.3 and alpha = 2, with V_T = 0.35 V.

In [7]:
v = np.linspace(0.45, 1.5, 400)
for alpha in (1.3, 2.0):
    d = dl.delay_vs_supply(v, 0.35, alpha) / dl.delay_vs_supply(1.2, 0.35, alpha)
    e = (v / 1.2) ** 2
    plt.plot(v, d * e, label=f"EDP, alpha = {alpha}")
    print(f"alpha {alpha}: EDP minimum at {v[np.argmin(d * e)]:.3f} V on the grid; 3VT/(3 - alpha) = {dl.edp_optimal_vdd(0.35, alpha):.3f} V")
plt.xlabel("VDD (V)"); plt.ylabel("EDP relative to 1.2 V"); plt.ylim(0, 3); plt.legend(); plt.show()

alpha 1.3: EDP minimum at 0.618 V on the grid; 3VT/(3 - alpha) = 0.618 V
alpha 2.0: EDP minimum at 1.050 V on the grid; 3VT/(3 - alpha) = 1.050 V


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).